# MicroVision Pipeline Results Analysis

This notebook analyzes the results of the semantic log dependency inference pipeline, including:
1.  **Sensitivity Analysis**: Examining how the similarity threshold affects Precision, Recall, and F1.
2.  **Comparative Analysis**: Comparing the Semantic model (Algorithm A) against a Syntactic Baseline (Algorithm B).
3.  **Thesis Summary**: Reviewing the formal documentation generated for the thesis.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Setup plotting style
plt.style.use('ggplot')
sns.set_palette("husl")

# Path configuration
DATA_FILE = Path("../data/sensitivity_analysis.csv")

if not DATA_FILE.exists():
    print(f"WARNING: {DATA_FILE} not found. Please run 'python scripts/sensitivity_analysis.py' first.")
else:
    print(f"Data found at {DATA_FILE}")

## 1. Sensitivity Analysis

We evaluate the trade-off between **Precision** and **Recall** by sweeping the `hybrid_score_threshold` from 0.0 to 1.0. 
*   **Low Threshold**: High Recall (finds many edges), Low Precision (lots of noise).
*   **High Threshold**: Low Recall (misses edges), High Precision (very confident).

The goal is to find the optimal F1 Score peak.

In [ ]:
if DATA_FILE.exists():
    df = pd.read_csv(DATA_FILE)
    
    # Create the visualization
    plt.figure(figsize=(12, 7))
    
    # Plot Metrics
    plt.plot(df['threshold'], df['precision'], label='Precision', marker='o', linestyle='--')
    plt.plot(df['threshold'], df['recall'], label='Recall', marker='s', linestyle='--')
    plt.plot(df['threshold'], df['f1'], label='F1 Score', marker='^', linewidth=3, color='black')

    # Formatting
    plt.title('Hyperparameter Sensitivity: Hybrid Score Threshold', fontsize=14)
    plt.xlabel('Threshold (Hybrid Score)', fontsize=12)
    plt.ylabel('Score', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.xlim(0, 1.0)
    plt.ylim(0, 1.1)

    # Highlight optimal point
    best_f1 = df['f1'].max()
    best_row = df[df['f1'] == best_f1].iloc[0]
    plt.annotate(f"Optimal F1={best_f1:.2f} @ {best_row['threshold']}", 
                 xy=(best_row['threshold'], best_f1), 
                 xytext=(best_row['threshold'], best_f1 + 0.1),
                 arrowprops=dict(facecolor='black', shrink=0.05))

    plt.show()
    
    # Display the table subset around the peak
    print("Optimal Configuration Window:")
    peak_idx = df['f1'].idxmax()
    range_view = slice(max(0, peak_idx - 2), min(len(df), peak_idx + 3))
    display(df.iloc[range_view])

## 2. Baseline Comparison

The table below summarizes the key differences between the traditional **Syntactic Approach (TF-IDF)** and the proposed **Semantic Approach (SBERT + RAG)**.

| Metric | Semantic Model (SBERT) | Baseline Model (TF-IDF) | Interpretation |
| :--- | :--- | :--- | :--- |
| **Raw Candidate Edges** | **629** | 2063 | Semantic model generated **3x less noise**. |
| **Unique Service Pairs** | 4 | 4 | Both models converged on the "dominant" interactions. |
| **Recall (Full GT)** | 10.5% | 10.5% | constrained by dataset sparsity; both found the same signal. |
| **Recall (Dominant)** | 100% | 100% | Both found the 2 main architectural links. |
| **Storage Footprint** | Low (Vector) | Low (Matrix) | Comparable. |
| **Candidates Filtered** | **Highly Efficient** | Low Efficiency | Semantic logic filters irrelevant matches early. |

### Conclusion
While top-line F1 scores are similar for this specific OpenStack dataset (due to its lexical simplicity), the **Semantic approach demonstrates superior specificity**, rejecting 70% more false candidates than the syntactic baseline. This property suggests SBERT will scale better to messy, heterogeneous production environments where syntactic overlap is high but semantic meaning differs.

## 3. Thesis Documentation

The findings above have been formalized into the thesis draft. Below is the current content of the comparative analysis section.

In [ ]:
from IPython.display import Markdown

THESIS_FILE = Path("../docs/THESIS_DRAFT.md")

if THESIS_FILE.exists():
    content = THESIS_FILE.read_text()
    display(Markdown(content))
else:
    print("Thesis draft not found.")